
# 📶 Conectividade Brasil — Dashboard

In [3]:

import os
import numpy as np
import pandas as pd
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from ipywidgets import interactive, HBox, VBox, IntSlider, SelectMultiple, Layout, Output
from IPython.display import display

# Caminho do CSV
DATA_PATH = Path(r"D:/OneDriveBackup/OneDrive - Claro SA/LAURA SILVA SOARES DE MELO/arquivos/projetos/projeto_conectividade/data/br_anatel_indice_brasileiro_conectividade_municipio.csv")

# Função para carregar dados
def load_data(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, sep=',', encoding='utf-8')
    df.columns = [c.strip().lower() for c in df.columns]
    return df

# Carregar dados
df = load_data(DATA_PATH)
nanos = sorted(df['ano'].dropna().unique())
ufs = sorted(df['sigla_uf'].dropna().unique())

# Funções auxiliares
def plot_bar_ibc(dfl, ano):
    ibc_uf = dfl.groupby('sigla_uf', as_index=False)['ibc'].mean().sort_values('ibc', ascending=False)
    return px.bar(ibc_uf, x='sigla_uf', y='ibc', color='ibc', color_continuous_scale='Blues', title=f"IBC médio por UF (ano {ano})")

def plot_line_evolucao(df, ufs_sel):
    if ufs_sel:
        df_evo = df[df['sigla_uf'].isin(list(ufs_sel))].groupby(['ano','sigla_uf'], as_index=False)['ibc'].mean()
        return px.line(df_evo, x='ano', y='ibc', color='sigla_uf', markers=True, title='Evolução do IBC por UF')
    else:
        df_evo = df.groupby('ano', as_index=False)['ibc'].mean()
        return px.line(df_evo, x='ano', y='ibc', markers=True, title='Evolução do IBC (média nacional)')

def plot_scatter(df, x_col, y_col, color_col, title):
    fig = px.scatter(df, x=x_col, y=y_col, color=color_col, trendline='ols', title=title)
    return fig

# Output para gráficos
out_plot = Output()

def render(nano_sel, ufs_sel, th_low_fibra, th_high_cob):
    with out_plot:
        out_plot.clear_output()  # Limpa gráficos anteriores
        dfl = df[df['ano'] == nano_sel].copy()
        if ufs_sel:
            dfl = dfl[dfl['sigla_uf'].isin(list(ufs_sel))]

        # KPIs
        print(f"IBC médio: {dfl['ibc'].mean():.1f}\nMunicípios: {len(dfl)}")

        # Gráficos
        display(plot_bar_ibc(dfl, nano_sel))
        display(plot_line_evolucao(df, ufs_sel))

        if {'fibra','ibc'}.issubset(dfl.columns):
            display(plot_scatter(dfl.dropna(subset=['fibra','ibc']), 'fibra', 'ibc', 'sigla_uf', 'Fibra vs IBC'))

        if {'adensamento_erbs','ibc'}.issubset(dfl.columns):
            display(plot_scatter(dfl.dropna(subset=['adensamento_erbs','ibc']), 'adensamento_erbs', 'ibc', 'sigla_uf', 'Adensamento ERBs vs IBC'))

        # Tabela prioritária
        if {'fibra','cobertura_pop_4g5g','ibc'}.issubset(dfl.columns):
            mask = (dfl['fibra'] <= th_low_fibra) & (dfl['cobertura_pop_4g5g'] >= th_high_cob)
            pri = dfl[mask].sort_values(['cobertura_pop_4g5g','ibc'], ascending=[False, True])
            display(pri.head(50))

# Widgets
ano_slider = IntSlider(description='Ano', min=int(min(nanos)), max=int(max(nanos)), value=int(max(nanos)), step=1, layout=Layout(width='50%'))
uf_select = SelectMultiple(options=ufs, description='UFs', layout=Layout(width='50%', height='200px'))
low_fibra = IntSlider(description='Baixa fibra (0-100)', min=0, max=100, value=33, step=1, layout=Layout(width='50%'))
high_cob = IntSlider(description='Alta cobertura (%)', min=0, max=100, value=80, step=1, layout=Layout(width='50%'))

display(VBox([HBox([ano_slider, uf_select]), HBox([low_fibra, high_cob])]))
out = interactive(render, nano_sel=ano_slider, ufs_sel=uf_select, th_low_fibra=low_fibra, th_high_cob=high_cob)
display(out_plot)
display(out)


Output()

interactive(children=(IntSlider(value=2024, description='Ano', layout=Layout(width='50%'), max=2024, min=2021)…